In [1]:
!pip install "numpy<2"
!pip install "scipy<1.13" "scikit-learn<1.4"
!pip install torch==2.2.2 torchvision torchaudio
!pip install open3d

!git clone https://github.com/isl-org/Open3D-ML.git
%cd Open3D-ML

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 50.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.

In [2]:
%%writefile run_track1.py
import numpy as np
import os
import glob
import zipfile
import open3d.ml as _ml3d
import open3d.ml.torch as ml3d

def setup_randlanet_model():
    print("Initializing model")
    cfg_file = "ml3d/configs/randlanet_semantickitti.yml"
    cfg = _ml3d.utils.Config.load_from_file(cfg_file)
    model = ml3d.models.RandLANet(**cfg.model)
    pipeline = ml3d.pipelines.SemanticSegmentation(model, device="gpu", **cfg.pipeline)

    ckpt_folder = "./logs/"
    os.makedirs(ckpt_folder, exist_ok=True)
    ckpt_path = os.path.join(ckpt_folder, "randlanet_semantickitti_202201071330utc.pth")
    randlanet_url = "https://storage.googleapis.com/open3d-releases/model-zoo/randlanet_semantickitti_202201071330utc.pth"

    if not os.path.exists(ckpt_path):
        print("Downloading weights")
        os.system(f"wget {randlanet_url} -O {ckpt_path}")

    pipeline.load_ckpt(ckpt_path=ckpt_path)
    print("Model ready")
    return pipeline

def load_velodyne_bin(bin_path):
    scan = np.fromfile(bin_path, dtype=np.float32)
    xyzi = scan.reshape((-1, 4))
    return xyzi

def run_actual_inference(pipeline, xyzi):
    xyz = xyzi[:, :3]
    dummy_labels = np.zeros(xyz.shape[0], dtype=np.int32)
    
    data = {'point': xyz, 'feat': None, 'label': dummy_labels}
    
    result = pipeline.run_inference(data)
    labels = result['predict_labels']
    scores = result['predict_scores']
    confidence = np.max(scores, axis=1) 
    
    return labels, confidence

if __name__ == "__main__":
    seg_pipeline = setup_randlanet_model()
    
    input_dir = '/kaggle/input/datasets/dipangsu/kitti-odometry-velodyne/dataset/sequences/08/velodyne'
    zip_filename = '/kaggle/working/SIH_Segmented_Seq08.zip'
    temp_path = '/kaggle/working/temp_frame.npy'
    
    bin_files = sorted(glob.glob(os.path.join(input_dir, '*.bin')))
    
    if len(bin_files) == 0:
        print("Files missing")
    else:
        print(f"Starting batch of {len(bin_files)} frames...")
        
        
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for count, bin_path in enumerate(bin_files):
                frame_id = os.path.splitext(os.path.basename(bin_path))[0]
                print(f"Processing {frame_id}")
                
                raw_xyzi = load_velodyne_bin(bin_path)
                predicted_labels, confidence_scores = run_actual_inference(seg_pipeline, raw_xyzi)
                
               
                xyz = raw_xyzi[:, :3]
                intensity = raw_xyzi[:, 3:4]
                labels_reshaped = predicted_labels.reshape(-1, 1).astype(np.float32)
                confidence_reshaped = confidence_scores.reshape(-1, 1).astype(np.float32)
                
                # Stack into 6 columns
                final_output = np.hstack((xyz, labels_reshaped, confidence_reshaped, intensity))
                
                
                np.save(temp_path, final_output)
                zipf.write(temp_path, arcname=f'{frame_id}_segmented.npy')
                
        
        if os.path.exists(temp_path):
            os.remove(temp_path)
                
        print("Process complete! ZIP file created safely.")

Writing run_track1.py


In [3]:
!python run_track1.py

Initializing model
--2026-09-04 02:24:12--  https://storage.googleapis.com/open3d-releases/model-zoo/randlanet_semantickitti_202201071330utc.pth
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.212.207, 74.125.134.207, 173.194.216.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.212.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5101179 (4.9M) [application/octet-stream]
Saving to: ‘./logs/randlanet_semantickitti_202201071330utc.pth’

./logs/randlanet_se 100%[===================>]   4.86M  --.-KB/s    in 0.07s   

2026-09-04 02:24:12 (68.1 MB/s) - ‘./logs/randlanet_semantickitti_202201071330utc.pth’ saved [5101179/5101179]

Model ready
Starting batch of 4071 frames...
Processing 000000
test 0/1:  93%|██████████████████████▎ | 82050/88109 [00:01<00:00, 67250.93it/s]/usr/local/lib/python3.12/dist-packages/open3d/_ml3d/torch/modules/metrics/semseg_metric.py:54: RuntimeWarning: Mean of empty slice
  accs.append